In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

df = spark.table("silver.combined_taxi_trips")

#prepare dataset
df_model = (
    df.select(
        "pulocationid",
        "dolocationid",
        "service_type",
        "pickup_datetime",
        "trip_time",
        "total_amount"
    )
    .dropna()
    .filter(
        (F.col("service_type") != "Unknown") &
        (F.col("pickup_datetime") > F.lit("2024-12-12"))
    )
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .withColumn("time_group", F.floor(F.col("pickup_hour") / 4))
)

#smaller sample to avoid serverless issues
df_model = df_model.sampleBy(
    "service_type",
    fractions={
        "Green": 0.3,
        "Uber": 0.001,
        "Lyft": 0.002,
        "Yellow": 0.002
    },
    seed=42
)


print("Rows after sampling:", df_model.count())
#index + features (same as your working version)
indexer = StringIndexer(
    inputCol="service_type",
    outputCol="service_type_idx",
    handleInvalid="keep"
)

df_model = indexer.fit(df_model).transform(df_model)

assembler = VectorAssembler(
    inputCols=["pulocationid", "dolocationid", "service_type_idx", "time_group"],
    outputCol="features"
)

df_model = assembler.transform(df_model)

#split
train, test = df_model.randomSplit([0.8, 0.2], seed=42)

#train models (same structure as your working RF code)
model_time = RandomForestRegressor(
    featuresCol="features",
    labelCol="trip_time"
).fit(train)

model_cost = RandomForestRegressor(
    featuresCol="features",
    labelCol="total_amount"
).fit(train)

#evaluate
for label, model in [("trip_time", model_time), ("total_amount", model_cost)]:
    preds = model.transform(test)

    rmse = RegressionEvaluator(labelCol=label, predictionCol="prediction", metricName="rmse").evaluate(preds)
    mae = RegressionEvaluator(labelCol=label, predictionCol="prediction", metricName="mae").evaluate(preds)
    r2 = RegressionEvaluator(labelCol=label, predictionCol="prediction", metricName="r2").evaluate(preds)

    print(f"\nLinear Regression results for {label}:")
    print({"rmse": rmse, "mae": mae, "r2": r2})